# Use gwen2.5-14b-instruct for judging top10 results from ft-reranker.ipynb

In [ ]:
!apt-get update && apt-get install -y zstd

!curl -fsSL https://ollama.com/install.sh | sh

import subprocess
import time
import os

if os.path.exists("/usr/local/bin/ollama") or os.path.exists("/usr/bin/ollama"):
    print("Ollama binary found! Starting two independent servers, one per T4 GPU...")

    base_env = os.environ.copy()
    base_env["OLLAMA_NUM_GPU"]      = "99"
    base_env["OLLAMA_NUM_PARALLEL"] = "2"

    # --- Instance 0: GPU 0, default port 11434 ---
    env0 = base_env.copy()
    env0["CUDA_VISIBLE_DEVICES"] = "0"
    env0["OLLAMA_HOST"]          = "0.0.0.0:11434"
    with open("ollama_gpu0.log", "w") as f:
        subprocess.Popen(["ollama", "serve"], stdout=f, stderr=f, env=env0)

    # --- Instance 1: GPU 1, port 11435 ---
    env1 = base_env.copy()
    env1["CUDA_VISIBLE_DEVICES"] = "1"
    env1["OLLAMA_HOST"]          = "0.0.0.0:11435"
    with open("ollama_gpu1.log", "w") as f:
        subprocess.Popen(["ollama", "serve"], stdout=f, stderr=f, env=env1)

    # Give both servers time to initialize

    import urllib.request

    def wait_for_ollama(host, timeout=60):
        url = f"{host}/api/tags"
        for _ in range(timeout):
            try:
                urllib.request.urlopen(url, timeout=1)
                print(f"{host} is ready.")
                return
            except:
                time.sleep(1)
        raise RuntimeError(f"{host} did not start within {timeout}s")

    wait_for_ollama("http://localhost:11434")
    wait_for_ollama("http://localhost:11435")

    print("Both Ollama servers are ready (GPU-0 → :11434, GPU-1 → :11435).")
else:
    print("Installation failed. Check the output above for errors.")


Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:3 https://cli.github.com/packages stable InRelease [3,917 B]
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:7 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,389 kB]
Get:8 https://cli.github.com/packages stable/main amd64 Packages [357 B]
Get:9 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:10 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [85.2 kB]
Get:11 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,924 kB]
Get:12 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:13 https://r2u.stat.illinois.edu/ubuntu jammy

In [ ]:
!OLLAMA_HOST=http://localhost:11434 ollama pull qwen2.5:14b-instruct
!OLLAMA_HOST=http://localhost:11435 ollama pull qwen2.5:14b-instruct


]11;?\pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest 
pulling 2049f5674b1e:   0% ▕                  ▏  22 MB/9.0 GB                  pulling manifest 
pulling 2049f5674b1e:   1% ▕                  ▏ 118 MB/9.0 GB                  pulling manifest 
pulling 2049f5674b1e:   2% ▕                  ▏ 167 MB/9.0 GB                  pulling manifest 
pulling 2049f5674b1e:   3% ▕                  ▏ 269 MB/9.0 GB                  pulling manifest 
pulling 2049f5674b1e:   4% ▕                  ▏ 374 MB/9.0 GB                  pulling manifest 
pulling 2049f5674b1e:   5% ▕                  ▏ 422 MB/9.0 GB                  pulling manifest 
pulling 2049f5674b1e:   6% ▕█                 ▏ 518 MB/9.0 GB                  pulling manifest 
pulling 2049f5674b1e:   7% ▕█                 ▏ 613 MB/9.0 GB                  pulling manifest 
pulling 

In [3]:
!pip install ollama

In [4]:
!pip install py_vncorenlp

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 20.6 MB/s eta 0:00:00
  Created wheel for py_vncorenlp: filename=py_vncorenlp-0.1.4-py3-none-any.whl size=4304 sha256=76d80cfaa764541d91fc25fb4d6359d20675c38eeba8de24d2c4750c4f749388
  Stored in directory: /root/.cache/pip/wheels/db/e5/ff/f4a1b4ece36e8582db1ca71150a34e987e65df50c35974e9bb
Successfully built py_vncorenlp


In [5]:
import json
import numpy as np
import pandas as pd
import py_vncorenlp

In [6]:
py_vncorenlp.download_model(save_dir='/kaggle/working')

--2026-03-08 07:59:44--  https://raw.githubusercontent.com/vncorenlp/VnCoreNLP/master/VnCoreNLP-1.2.jar
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.109.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 27412703 (26M) [application/octet-stream]
Saving to: ‘VnCoreNLP-1.2.jar’

     0K .......... .......... .......... .......... ..........  0% 4.29M 6s
    50K .......... .......... .......... .......... ..........  0% 9.46M 4s
   100K .......... .......... .......... .......... ..........  0% 6.71M 4s
   150K .......... .......... .......... .......... ..........  0% 23.4M 3s
   200K .......... .......... .......... .......... ..........  0% 25.6M 3s
   250K .......... .......... .......... .......... ..........  1% 7.22M 3s
   300K .......... .......... .......... .......... ..........  1% 87.9M 3s
   350K ..

In [7]:
# Load the word and sentence segmentation component
rdrsegmenter = py_vncorenlp.VnCoreNLP(annotators=["wseg"], save_dir='/kaggle/working')

text = "Ông Nguyễn Khắc Chúc  đang làm việc tại Đại học Quốc gia Hà Nội. Bà Lan, vợ ông Chúc, cũng làm việc tại đây."

output = rdrsegmenter.word_segment(text)

print(output)
# ['Ông Nguyễn_Khắc_Chúc đang làm_việc tại Đại_học Quốc_gia Hà_Nội .', 'Bà Lan , vợ ông Chúc , cũng làm_việc tại đây .']

2026-03-08 07:59:49 INFO  WordSegmenter:24 - Loading Word Segmentation model
['Ông Nguyễn_Khắc_Chúc đang làm_việc tại Đại_học Quốc_gia Hà_Nội .', 'Bà Lan , vợ ông Chúc , cũng làm_việc tại đây .']


In [8]:
with open("/kaggle/input/datasets/duongquanganh/privatetest/DRILL_PrivateTest/private_test.json","r",encoding = "utf-8") as f:
    file = json.load(f)

qid_2_text = {}
text_2_qid = {}
for q in file:
    output = rdrsegmenter.word_segment(q['question'])
    q['question'] = " ".join(output)
    qid_2_text[q['qid']] = q['question']
    text_2_qid[q['question']] = q['qid']

In [9]:
# segment laws in corpus
corpus_Id2Text = {}
corpus_Text2Id = {}


with open("/kaggle/input/datasets/duongquanganh/chunked-corpus/chunked_corpus.json", "r", encoding="utf-8") as f:
    corpus = json.load(f)

for raw in corpus:
    corpus_Id2Text[raw['chunk_id']] = raw['content_Article']
    corpus_Text2Id[raw['content_Article']] = raw['chunk_id']

In [10]:
with open("/kaggle/input/datasets/quanganhiuhin/top10-r0-7699/reranked_top10_results_input_judge 0.7699.json", "r", encoding="utf-8") as f:
    top10 = json.load(f)

for q in top10:
    relevant_laws_text = []
    for rel in q['relevant_laws']:
        relevant_laws_text.append(corpus_Id2Text[rel])
    q['relevant_laws'] = relevant_laws_text

In [11]:
import ollama

# Let's test the 32B model
try:
    response = ollama.chat(model='qwen2.5:14b-instruct', messages=[
        {'role': 'user', 'content': 'Explain the concept of "quantization" in LLMs like a five-year-old.'}
    ])
    print("--- Response from Qwen 2.5 14B ---")
    print(response['message']['content'])
except Exception as e:
    print(f"An error occurred: {e}")
    print("Tip: If it says 'connection refused', the background server might have crashed due to VRAM limits.")

--- Response from Qwen 2.5 14B ---
Hey there little one! Let's talk about something fun and magical called "quantization" in big computer brains like LLMs (that's short for Large Language Models).

Imagine you have a big box of crayons with lots of colors. Now, these crayons represent all the tiny bits of information in a computer brain. But what if we want to make our crayon box smaller and easier to carry around? We can do that by picking just a few crayons that are super important to us.

That's kind of like what quantization does! In a computer brain, quantization means taking all those tiny bits of information and rounding them down to smaller, simpler numbers. It's like picking the most important crayons to keep and getting rid of the ones we don't use as much.

This makes the computer brain smaller and faster, like having a smaller crayon box that's easier to take everywhere! It also helps the computer brain use less power, like how carrying fewer crayons makes your backpack lig

In [12]:
import ollama

# One persistent client per server instance keeps connections warm
_clients = {
    "http://localhost:11434": ollama.Client(host="http://localhost:11434"),
    "http://localhost:11435": ollama.Client(host="http://localhost:11435"),
}

def judge(question, answer, host="http://localhost:11434"):
    system_instruction = (
        "Bạn là một trợ lý kiểm định văn bản pháp luật chuyên nghiệp. "
        "Nhiệm vụ của bạn là phân loại tài liệu dựa trên câu hỏi pháp lý. "
        "Bạn PHẢI trả về kết quả dưới định dạng JSON nguyên bản, không kèm theo lời dẫn giải văn bản nào khác."
    )

    prompt = f"""
    ### VẤN ĐỀ PHÁP LÝ:
    {question}
    
    ### NỘI DUNG TÀI LIỆU TRÍCH XUẤT:
    {answer}
    
    ### QUY TẮC QUYẾT ĐỊNH:
    1. "KEEP": Nếu tài liệu trả lời trực tiếp vấn đề pháp lý.
    2. "DROP": Nếu tài liệu không liên quan, chung chung hoặc không trùng khớp đối tượng.
    
    ### YÊU CẦU ĐẦU RA (JSON ONLY):
    Trả về một đối tượng JSON với các trường sau:
    - "reasoning": Phân tích ngắn gọn lý do tại sao giữ hoặc loại.
    - "decision": Chỉ ghi "KEEP" hoặc "DROP".
    - "confidence_score": Độ tự tin của bạn (từ 0.0 đến 1.0).
    
    Mẫu:
    {{
      "reasoning": "...",
      "decision": "...",
      "confidence_score": 0.0
    }}
    """

    response = _clients[host].chat(
        model='qwen2.5:14b-instruct',
        messages=[
            {'role': 'system', 'content': system_instruction},
            {'role': 'user', 'content': prompt}
        ],
        format='json',
        options={
            "temperature": 0.0,
            "top_p": 0.1,
            #"num_ctx": 2048   # cap context to shrink KV-cache
        }
    )
    return response.message.content


In [ ]:
import threading
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime

NUM_WORKERS = 4
OLLAMA_HOSTS = ["http://localhost:11434", "http://localhost:11435"]

# Map thread name → (worker label, assigned host)
_thread_info = {}
_map_lock = threading.Lock()
_worker_counter = 0

def get_worker_info():
    """Assign a stable label and a GPU host to the calling thread (round-robin)."""
    global _worker_counter
    tid = threading.current_thread().name
    with _map_lock:
        if tid not in _thread_info:
            slot = _worker_counter
            _worker_counter += 1
            host = OLLAMA_HOSTS[slot % len(OLLAMA_HOSTS)]
            _thread_info[tid] = (f"W-{slot} (GPU-{slot % len(OLLAMA_HOSTS)})", host)
        return _thread_info[tid]

def log(worker, msg):
    ts = datetime.now().strftime("%H:%M:%S.%f")[:-3]
    print(f"[{ts}] [{worker}] {msg}", flush=True)

def process_query(q):
    """Judge all candidates for one query on its assigned GPU instance."""
    worker, host = get_worker_info()
    qid = q['qid']
    candidates = q['relevant_laws']

    log(worker, f"START  qid={qid}  ({len(candidates)} candidates)")
    t0 = time.time()

    judged_candidates = []
    for idx, rel in enumerate(candidates, 1):
        t_c = time.time()
        res_str = judge(question=qid_2_text[qid], answer=rel, host=host)
        res_json = json.loads(res_str)
        decision = res_json['decision']
        elapsed = time.time() - t_c
        log(worker, f"  candidate {idx}/{len(candidates)} → {decision}  ({elapsed:.1f}s)")
        if decision == 'KEEP':
            judged_candidates.append(rel)

    total = time.time() - t0
    log(worker, f"DONE   qid={qid}  kept={len(judged_candidates)}  total={total:.1f}s")
    return {'qid': qid, 'relevant_laws': judged_candidates}

# --- Main loop ---
subset = top10   # process all questions; slice (e.g. top10[:6]) to test

judged_top10 = [None] * len(subset)

print(f"Processing {len(subset)} questions with {NUM_WORKERS} workers (2 per GPU, 2 Ollama slots each)...\n")
with ThreadPoolExecutor(max_workers=NUM_WORKERS) as pool:
    future_to_idx = {pool.submit(process_query, q): i for i, q in enumerate(subset)}
    completed = 0
    for future in as_completed(future_to_idx):
        idx = future_to_idx[future]
        result = future.result()
        judged_top10[idx] = result
        completed += 1
        print(f"\n>>> Overall progress: {completed}/{len(subset)} questions done\n", flush=True)

print("\nAll done.")
judged_top10


Processing 627 questions with 4 workers (2 per GPU, 2 Ollama slots each)...

[08:01:21.517] [W-0 (GPU-0)] START  qid=1497  (10 candidates)
[08:01:21.518] [W-1 (GPU-1)] START  qid=5107  (10 candidates)
[08:01:21.518] [W-2 (GPU-0)] START  qid=4960  (10 candidates)
[08:01:21.519] [W-3 (GPU-1)] START  qid=16735  (10 candidates)
[08:01:34.890] [W-2 (GPU-0)]   candidate 1/10 → KEEP  (13.4s)
[08:01:36.443] [W-0 (GPU-0)]   candidate 1/10 → KEEP  (14.9s)
[08:01:39.567] [W-1 (GPU-1)]   candidate 1/10 → KEEP  (18.0s)
[08:01:39.692] [W-3 (GPU-1)]   candidate 1/10 → DROP  (18.2s)
[08:01:45.289] [W-0 (GPU-0)]   candidate 2/10 → KEEP  (8.8s)
[08:01:47.063] [W-2 (GPU-0)]   candidate 2/10 → DROP  (12.2s)
[08:01:51.334] [W-1 (GPU-1)]   candidate 2/10 → KEEP  (11.8s)
[08:01:55.187] [W-3 (GPU-1)]   candidate 2/10 → DROP  (15.5s)
[08:01:55.869] [W-0 (GPU-0)]   candidate 3/10 → DROP  (10.6s)
[08:02:01.763] [W-2 (GPU-0)]   candidate 3/10 → DROP  (14.7s)
[08:02:03.318] [W-1 (GPU-1)]   candidate 3/10 → DROP  (

KeyError: 'decision'

In [ ]:
# Save to JSON file
with open("judged_top10_results_submit_for_check.json", "w", encoding="utf-8") as f:
    json.dump(judged_top10, f, ensure_ascii=False, indent=2)

print(f"Saved judged top 10 reranked results for {len(judged_top10)} questions to 'judged_top10_results_submit_for_check.json'")

In [ ]:
submit_judged_top10 = []
for q in judged_top10:
    submit_judged_top10.append({
        'qid': q['qid'],
        'relevant_laws': [int(corpus_Text2Id[rel].split("_")[0]) for rel in q['relevant_laws']]
    })


# Save to JSON file
with open("judged_top10_results_submit_for_submit.json", "w", encoding="utf-8") as f:
    json.dump(submit_judged_top10, f, ensure_ascii=False, indent=2)

print(f"Saved judged top 10 reranked results for {len(submit_judged_top10)} questions to 'judged_top10_results_submit_for_submit.json'")

In [ ]:
submit_judged_top10